# 03. Attack grid: attack success under the unconstrained and constrained attackers (RQ1, RQ2)

This notebook turns the saved attack grid into the study's attack-success results:

- **Tables IV and S-XI** (`reports/table2_robust.csv`): mean attack success rate (ASR) per model, attack and threat model, averaged over the datasets each model ran on, at $\epsilon = 0.25$ (Table IV) and at every budget (Table S-XI).
- **Tables S-V to S-X** (`reports/tables/grid_per_dataset.csv`): every attack, model and budget, per dataset.
- **Figures S1 and S2** (`fig_robust_curves_{constrained,unconstrained}`): worst-case ASR against the budget.
- The other grid summaries in `reports/` (`table2b_robust_<dataset>.md`, `table3_attacks.md`) and the numbers behind the RQ1 and RQ2 statements (`scripts/check_claims.py`).

**Inputs:** the attack grid of the six datasets, `runs/<dataset>/attacks/results*.csv` (written by `fraudadv.attacks.run_attacks`). The live cell also reads the saved ULB models (`runs/ulb/mlp`, `runs/ulb/xgboost`), the grid's thresholds and fraud sample (`runs/ulb/attacks/tau.json`, `sample.npz`, `attack_scores_full_all.npz`) and `data/processed/ulb`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

## The attack grid

`make_report.load_attacks` reads every row of the grid: one row per dataset, model, threat model (`mode`), attack and budget `eps`, with the ASR, the dollar-weighted ASR and the ranking metrics before and after the attack. Every table and figure below starts from this frame.

In [ ]:
(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)     # the output folders, as make_results.sh makes them

from fraudadv.report import make_report as MR
from fraudadv.report import tables

A = MR.load_attacks()          # runs/<dataset>/attacks/results*.csv of all datasets
print(len(A), "rows; models per dataset:", A.groupby("dataset")["model"].nunique().to_dict())
A.head()

## Tables IV and S-XI: the attack survey

`make_report.table_robust` writes the mean ASR and dollar ASR of every model, attack and threat model at $\epsilon \in \{0.1, 0.25, 0.5\}$, averaged over the datasets each model ran on (`reports/table2_robust.{csv,md}`): Table IV is its $\epsilon = 0.25$ column, Table S-XI all of it. It also writes the same per dataset with the ranking metrics (`table2b_robust_<dataset>.md`). `make_report.table_attacks` writes the catalogue of attacks (`table3_attacks.md`).

In [ ]:
MR.table_robust(A)
MR.table_attacks()
nb.show_csv("reports/table2_robust.csv", rows=10)
print((ROOT / "reports" / "table3_attacks.md").read_text())

## Figures S1 and S2: worst-case ASR against the budget

`make_report.fig_curves` draws the worst case over all attacks against $\epsilon$, one panel per dataset and one line per model, for the constrained attacker (Figure S1) and the unconstrained one (Figure S2).

In [ ]:
for mode in ("constrained", "unconstrained"):
    MR.fig_curves(A, mode)          # reports/fig_robust_curves_<mode>.{png,pdf}
nb.show_figure("fig_robust_curves_constrained")

In [ ]:
nb.show_figure("fig_robust_curves_unconstrained")

## Tables S-V to S-X: the grid per dataset

`tables.grid_per_dataset` writes `reports/tables/grid_per_dataset.csv`: ASR by dataset, threat model, attack, model and budget, with the robust AUC and KS under the minimum-effort attack. Shown here: the ULB rows at $\epsilon = 0.25$ (Table S-V shows every budget).

In [ ]:
P = tables.grid_per_dataset(A)  # reports/tables/grid_per_dataset.csv
P[(P["dataset"] == "ulb") & (P["eps"] == 0.25)].pivot_table(index=["mode", "attack"], columns="model", values="asr").round(3)

## The numbers behind the RQ1 and RQ2 statements

`scripts/check_claims.py` prints, from the same grid, the numbers that the results section quotes in words (Sections VI-B and VI-C): worst-case ASR by budget, the constrained/unconstrained ratios and their medians by model class, and the attack-by-attack comparisons (white-box against Square and HopSkipJump, the gain of the worst case over the best single attack). It writes nothing.

In [ ]:
import runpy

runpy.run_path(str(ROOT / "scripts" / "check_claims.py"), run_name="__main__");

## Run it yourself: APGD and AATS on 20 ULB frauds

This cell attacks again, live, a few frauds of the grid with the grid's own functions: APGD (`fraudadv.attacks.blackwhite.apgd`) on the saved MLP and AATS (`fraudadv.attacks.blackwhite.aats`) on the saved XGBoost, under the constrained threat model at $\epsilon = 0.25$. It uses the grid's 1%-FPR thresholds (`tau.json`) and its fraud sample (`sample.npz`), takes the first 20 frauds of the sample that each model detects, and compares each fraud's outcome with the attacked score the grid stored (`attack_scores_full_all.npz`). On ULB the constrained attacker controls only the amount and the hour. It takes about 10 seconds and writes nothing.

The ASR here is an illustration on 20 frauds, not the study's number. The study's number is the grid's ASR over all the frauds of the sample the model detects, shown in the second table. Two differences from the grid: APGD's random starts depend on the batch it is given, and the grid runs AATS once at the largest budget and counts the evasions that cost at most 0.25. A fraud's live outcome could therefore differ from the grid's.

In [ ]:
import json

import numpy as np
import pandas as pd

from fraudadv.attacks.blackwhite import aats, apgd
from fraudadv.attacks.constraints import Constraints
from fraudadv.attacks.run_attacks import ATTACK_AATS, ATTACK_PGD, load_model
from fraudadv.data.load import load

DATASET, EPS, N = "ulb", 0.25, 20
a, meta = load(DATASET)
attack_dir = ROOT / "runs" / DATASET / "attacks"
tau = json.loads((attack_dir / "tau.json").read_text())       # the grid's 1%-FPR thresholds (logits)
S = np.load(attack_dir / "sample.npz")["S"]                   # the grid's shared fraud sample (test rows)
X0 = a["X_test"][S]
cons = Constraints(meta, a["X_train"], "constrained")
print("attacker-controllable features:", [meta["features"][j] for j in cons.mutable])
stored = np.load(attack_dir / "attack_scores_full_all.npz")   # the grid's attacked score of every fraud

rows = []
for fam, attack in (("mlp", ATTACK_PGD), ("xgboost", ATTACK_AATS)):
    model = load_model(DATASET, fam)
    detected = np.flatnonzero(model.logit(X0) > tau[fam])[:N]     # the first N frauds of the sample the model detects
    if fam == "mlp":
        X_adv, logit_adv = apgd(model, X0[detected], tau[fam], EPS, cons)
    else:
        X_adv, logit_adv, _ = aats(model, X0[detected], tau[fam], EPS, cons)
    evaded = logit_adv < tau[fam]
    position = {row: k for k, row in enumerate(stored[f"{fam}|constrained|{attack}|rows"])}
    in_grid = np.array([stored[f"{fam}|constrained|{attack}|{EPS}"][position[i]] < tau[fam] for i in detected])
    rows.append({"model": fam, "attack": attack, "frauds attacked": len(detected), "evaded (live)": int(evaded.sum()),
                 "ASR (live)": evaded.mean(), "evaded in the grid": int(in_grid.sum()),
                 "same outcome as the grid": f"{int((evaded == in_grid).sum())} of {len(detected)}",
                 "largest L-inf move": float(cons.linf(X_adv, X0[detected]).max())})
display(pd.DataFrame(rows))

# the study's numbers: the grid's ASR at eps 0.25 over every detected fraud of the sample
grid_rows = A[(A["dataset"] == DATASET) & (A["mode"] == "constrained") & (A["eps"] == EPS)
              & (((A["model"] == "mlp") & (A["attack"] == ATTACK_PGD)) | ((A["model"] == "xgboost") & (A["attack"] == ATTACK_AATS)))]
grid_rows[["model", "attack", "eps", "n_detected", "asr"]]

## Rerun the runs behind these results from scratch

The cell below holds the calls that `scripts/reproduce.sh` makes for the attack grid of one dataset, with the study's options: black-box attacks on the whole fraud sample (`full_bb=True`) and an hour of exact MILP per tree model and threat model (`milp_budget=3600`). The foundation models are attacked in a second run, on 240 random frauds of the sample (all of them on Sparkov). This takes hours of CPU per dataset (days on the large ones). It writes into `runs/<dataset>/attacks/` and replaces the saved results, so it is off here (`RERUN = False`). Afterwards, run this notebook again from the top.

In [ ]:
RERUN = False
DATASET = "ulb"            # ulb | ieee_cis | sparkov | tabformer | paysim | baf

if RERUN:
    from fraudadv.attacks import run_attacks

    TAB = ["xgboost", "catboost", "mlp", "ft_transformer", "ivis"]         # tabular families (every dataset)
    CTX = [] if DATASET in ("ulb", "baf") else ["hgnn", "hgt"]            # graph models: card/account sequences only
    FMS = [] if DATASET in ("ulb", "baf") else ["chronos2", "timesfm"]    # foundation models, likewise
    if FMS:
        import chronos  # noqa: F401  must be imported before TensorFlow (ivis loads it): the reverse order aborts
    run_attacks.run(DATASET, TAB + CTX, modes=("unconstrained", "constrained"), full_bb=True, milp_budget=3600,
                    tag="_full_all")
    if FMS:
        run_attacks.run(DATASET, FMS, modes=("unconstrained", "constrained"), full_bb=True, milp_budget=3600,
                        fm_full_n=100000 if DATASET == "sparkov" else 240, tag="_full_fm")

## Check against the reference

Every file this notebook wrote under `reports/`, compared byte for byte with the reference build (the folder in `FRAUDADV_REFERENCE`, holding a previous build of `reports/`).

In [ ]:
DATASETS = ["ulb", "ieee_cis", "sparkov", "tabformer", "paysim", "baf"]
nb.check(["reports/table2_robust.csv", "reports/table2_robust.md"]
         + [f"reports/table2b_robust_{d}.md" for d in DATASETS]
         + ["reports/table3_attacks.md", "reports/tables/grid_per_dataset.csv"]
         + [f"reports/fig_robust_curves_{mode}.{ext}" for mode in ("constrained", "unconstrained") for ext in ("pdf", "png")]);